In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)

ds = load_dataset("ai4bharat/Lahaja", split="test", token=token)
print(ds); print(ds.column_names)
for r in ds.select(range(10)):
    print({k: v for k, v in r.items() if k != "audio"})

In [2]:
from huggingface_hub import HfApi
print("Lahaja revision:", HfApi().dataset_info("ai4bharat/Lahaja", token=token).sha)

meta = ds.remove_columns(["audio_filepath"]).to_pandas()
assert meta["fname"].is_unique
meta = meta[meta["duration"] <= 30]                      # Whisper's limit
sample = (meta.sample(frac=1, random_state=42)           # shuffle
              .groupby("sp_id").head(8))                 # max 8 per speaker
print(len(sample), sample["sp_id"].nunique())
print(sample["scenario"].value_counts())
sample[["fname","sp_id","scenario","native_language","age_group","gender","duration"]] \
    .to_csv("lahaja_sample_v1.csv", index=False)

In [3]:
!pip install -q jiwer librosa soundfile accelerate

In [4]:
import re, unicodedata, pandas as pd, jiwer, numpy as np

TEXT_COL  = "text"
AUDIO_COL = "audio_filepath"
SR = 16000

def normalize(t):
    t = unicodedata.normalize("NFC", t or "")
    t = t.replace("\u200d", "").replace("\u200c", "")      # ZWJ / ZWNJ
    # keep letters (L*), combining marks (M*), numbers (N*), whitespace
    t = "".join(c if (unicodedata.category(c)[0] in "LMN" or c.isspace()) else " " for c in t)
    return re.sub(r"\s+", " ", t).strip()

def get_audio(ex):
    """Return float32 mono 16 kHz numpy array from AudioDecoder or dict."""
    a = ex[AUDIO_COL]
    if hasattr(a, "get_all_samples"):                       # torchcodec decoder
        s = a.get_all_samples()
        wav, sr = s.data.float().mean(0).numpy(), s.sample_rate
    else:                                                   # dict style
        wav, sr = np.asarray(a["array"], dtype=np.float32), a["sampling_rate"]
    if sr != SR:
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)

def score(refs, preds):
    r = [normalize(x) for x in refs]
    p = [normalize(x) for x in preds]
    return {"WER": jiwer.wer(r, p), "CER": jiwer.cer(r, p)}

def breakdown(df, col):
    rows = []
    for g, d in df.groupby(col):
        s = score(d["reference"], d["prediction"])
        rows.append({col: g, "n": len(d), **s})
    return pd.DataFrame(rows).sort_values("WER")

# quick sanity check
wav = get_audio(ds[0]); print(wav.shape, wav.dtype, len(wav)/SR, "sec")
print(normalize(ds[0][TEXT_COL]))

In [5]:
!pip uninstall -y torchao
!pip install -q -U qwen-asr jiwer librosa soundfile

In [6]:
import torch, os, time, numpy as np, pandas as pd
from qwen_asr import Qwen3ASRModel

MODEL_ID = "shujaAK/Hindi-ASR-Qwen3-0.6B-Finetuned"

model = Qwen3ASRModel.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,          # T4 has no fast bf16; the model card also uses fp16
    device_map="cuda:0",
    max_inference_batch_size=16,
    max_new_tokens=256,
)

durs = np.array(ds["duration"])
print("max duration:", durs.max(), "| >30s:", (durs > 30).sum())

In [7]:
from huggingface_hub import snapshot_download
import torch

print("cuda available:", torch.cuda.is_available())   # must be True, see note below

MODEL_ID = "shujaAK/Hindi-ASR-Qwen3-0.6B-Finetuned"
local_dir = snapshot_download(MODEL_ID)
print(local_dir)

from qwen_asr import Qwen3ASRModel
model = Qwen3ASRModel.from_pretrained(
    local_dir,
    dtype=torch.float16,
    device_map="cuda:0",
    max_inference_batch_size=16,
    max_new_tokens=256,
)

durs = np.array(ds["duration"])
print("max duration:", durs.max(), "| >30s:", (durs > 30).sum())

In [8]:
import transformers, huggingface_hub, torch
print("transformers", transformers.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [9]:
!pip install -q "huggingface_hub>=0.34,<1.0"

In [10]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)
ds = load_dataset("ai4bharat/Lahaja", split="test", token=token)

In [11]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

print(ds)   # should show 6152 rows

In [12]:
import re, unicodedata, pandas as pd, jiwer, numpy as np

TEXT_COL  = "text"
AUDIO_COL = "audio_filepath"
SR = 16000

def normalize(t):
    t = unicodedata.normalize("NFC", t or "")
    t = t.replace("\u200d", "").replace("\u200c", "")   # ZWJ / ZWNJ
    t = t.replace("\u093c", "")                          # nukta
    t = t.lower()
    t = "".join(c if (unicodedata.category(c)[0] in "LMN" or c.isspace()) else " " for c in t)
    return re.sub(r"\s+", " ", t).strip()

def get_audio(ex):
    a = ex[AUDIO_COL]
    if hasattr(a, "get_all_samples"):
        s = a.get_all_samples()
        wav, sr = s.data.float().mean(0).numpy(), s.sample_rate
    else:
        wav, sr = np.asarray(a["array"], dtype=np.float32), a["sampling_rate"]
    if sr != SR:
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)

def score(refs, preds):
    r = [normalize(x) for x in refs]
    p = [normalize(x) for x in preds]
    return {"WER": jiwer.wer(r, p), "CER": jiwer.cer(r, p)}

def breakdown(df, col):
    rows = []
    for g, d in df.groupby(col):
        rows.append({col: g, "n": len(d), **score(d["reference"], d["prediction"])})
    return pd.DataFrame(rows).sort_values("WER")

# sanity checks
print(normalize("ख़रीदना फ़िल्म ट्रांसफ़र।") == normalize("खरीदना फिल्म ट्रांसफर?"))  # True
wav = get_audio(ds[0]); print(wav.shape, len(wav)/SR, "sec")                        # ~11.6 sec

In [13]:
import transformers, huggingface_hub
print(transformers.__version__, huggingface_hub.__version__)

In [14]:
!hf download shujaAK/Hindi-ASR-Qwen3-0.6B-Finetuned --local-dir /kaggle/working/qwen_ft

import os
os.makedirs("/kaggle/working/qwen_ft/additional_chat_templates", exist_ok=True)
print(os.listdir("/kaggle/working/qwen_ft"))

In [15]:
import torch, numpy as np
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    "/kaggle/working/qwen_ft",
    dtype=torch.float16,
    device_map="cuda:0",
    max_inference_batch_size=16,
    max_new_tokens=256,
)

durs = np.array(ds["duration"])
print("max duration:", durs.max(), "| >30s:", (durs > 30).sum())

In [16]:
def transcribe_batch(wavs):
    res = model.transcribe(audio=[(w, SR) for w in wavs], language="Hindi")
    return [(r.text if hasattr(r, "text") else str(r)) for r in res]

idx = list(range(8))
preds = transcribe_batch([get_audio(ds[i]) for i in idx])
for i, p in zip(idx, preds):
    print("REF:", ds[i][TEXT_COL]); print("HYP:", p); print()

In [17]:
import os, time
OUT = "predictions_qwen.csv"
BS = 16

done = set(pd.read_csv(OUT)["idx"]) if os.path.exists(OUT) else set()
order = [int(i) for i in np.argsort(durs) if int(i) not in done]
print("remaining:", len(order))

buf, t0 = [], time.time()
for s in range(0, len(order), BS):
    b = order[s:s+BS]
    hyp = transcribe_batch([get_audio(ds[i]) for i in b])
    for i, h in zip(b, hyp):
        r = ds[i]
        buf.append({"idx": i, "fname": r["fname"], "reference": r[TEXT_COL], "prediction": h})
    if (s // BS) % 10 == 0 or s + BS >= len(order):
        pd.DataFrame(buf).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        buf = []
        print(f"{s+len(b)}/{len(order)}  {time.time()-t0:.0f}s")

In [18]:
res = pd.read_csv(OUT).fillna("")
meta = ds.remove_columns(AUDIO_COL).to_pandas().reset_index().rename(columns={"index": "idx"})
res = res.merge(meta.drop(columns=["text"]), on="idx")

print(score(res["reference"], res["prediction"]))
for c in ["native_language", "scenario", "gender", "age_group"]:
    print(breakdown(res, c).to_string(index=False), "\n")

print("preds containing digits:", res["prediction"].str.contains(r"\d").sum())
print("preds containing Latin letters:", res["prediction"].str.contains(r"[A-Za-z]").sum())

In [19]:
r = [normalize(x) for x in res["reference"]]
p = [normalize(x) for x in res["prediction"]]
o = jiwer.process_words(r, p)
N = sum(len(x.split()) for x in r)
print({"sub": o.substitutions/N, "del": o.deletions/N, "ins": o.insertions/N})

res["ref_len"] = [len(x.split()) for x in r]
res["hyp_len"] = [len(x.split()) for x in p]
res["ratio"] = res["hyp_len"] / res["ref_len"].clip(lower=1)
print("empty preds:", (res["hyp_len"] == 0).sum(),
      "| ratio>2:", (res["ratio"] > 2).sum(),
      "| ratio<0.5:", (res["ratio"] < 0.5).sum())

res["sample_wer"] = [jiwer.wer(a, b) if a else np.nan for a, b in zip(r, p)]
print(res["sample_wer"].describe())
print(res.sort_values("sample_wer", ascending=False)[["reference", "prediction", "sample_wer"]].head(10).to_string())

In [20]:
def collapse_repeats(text, max_n=10, max_rep=2):
    """Collapse any word n-gram repeated more than max_rep times in a row."""
    w = text.split()
    changed = True
    while changed:
        changed = False
        for n in range(1, max_n + 1):
            i = 0
            while i + n * (max_rep + 1) <= len(w):
                blk, reps = w[i:i+n], 1
                while w[i+reps*n : i+(reps+1)*n] == blk:
                    reps += 1
                if reps > max_rep:
                    del w[i + max_rep*n : i + reps*n]
                    changed = True
                i += 1
    return " ".join(w)

p_norm = [normalize(x) for x in res["prediction"]]
p_clean = [collapse_repeats(x) for x in p_norm]
res["prediction_clean"] = p_clean
res["looped"] = [a != b for a, b in zip(p_norm, p_clean)]

r_norm = [normalize(x) for x in res["reference"]]
print("looped clips:", res["looped"].sum())
print("strict  :", score(res["reference"], res["prediction"]))
print("collapsed:", score(res["reference"], res["prediction_clean"]))

# where does the error concentrate?
res["ref_words"] = [len(x.split()) for x in r_norm]
res["len_bin"] = pd.cut(res["ref_words"], [0, 2, 5, 10, 20, 200])
print(breakdown(res, "len_bin").to_string(index=False))
print(res.groupby("looped")["duration"].describe())

In [21]:
import os
print(os.path.exists("predictions_qwen.csv"), os.path.getsize("predictions_qwen.csv"))

In [22]:
import numpy as np, pandas as pd, jiwer, json, platform, torch, transformers

def breakdown(df, col):
    rows = [{col: g, "n": len(d), **score(d["reference"], d["prediction"])}
            for g, d in df.groupby(col, observed=True)]
    return pd.DataFrame(rows).sort_values("WER")

OUT = "predictions_qwen.csv"
res = pd.read_csv(OUT).fillna("")
print("rows:", len(res), "unique idx:", res["idx"].nunique())

meta = ds.remove_columns(AUDIO_COL).to_pandas().reset_index().rename(columns={"index": "idx"})
res = res.merge(meta.drop(columns=["text"]), on="idx")

overall = score(res["reference"], res["prediction"])
print(overall)
for c in ["native_language", "scenario", "gender", "age_group"]:
    print(breakdown(res, c).to_string(index=False), "\n")
print("preds with digits:", res["prediction"].str.contains(r"\d").sum())
print("preds with Latin letters:", res["prediction"].str.contains(r"[A-Za-z]").sum())

json.dump({
    "model": "Hindi-ASR-Qwen3-0.6B-Finetuned", "n_samples": len(res), **overall,
    "decoding": "qwen-asr default, language=Hindi forced",
    "normalization": "NFC, strip ZWJ/ZWNJ, strip nukta, lowercase, drop non L/M/N chars",
    "versions": {"transformers": transformers.__version__, "torch": torch.__version__,
                 "jiwer": jiwer.__version__, "python": platform.python_version()},
}, open("metrics_qwen.json", "w"), indent=2, ensure_ascii=False)

In [23]:
def collapse_repeats(text, max_n=10, max_rep=2):
    w = text.split()
    changed = True
    while changed:
        changed = False
        for n in range(1, max_n + 1):
            i = 0
            while i + n * (max_rep + 1) <= len(w):
                blk, reps = w[i:i+n], 1
                while w[i+reps*n : i+(reps+1)*n] == blk:
                    reps += 1
                if reps > max_rep:
                    del w[i + max_rep*n : i + reps*n]
                    changed = True
                i += 1
    return " ".join(w)

r = [normalize(x) for x in res["reference"]]
p = [normalize(x) for x in res["prediction"]]

o = jiwer.process_words(r, p)
N = sum(len(x.split()) for x in r)
print({"sub": o.substitutions/N, "del": o.deletions/N, "ins": o.insertions/N})

res["ref_len"] = [len(x.split()) for x in r]
res["hyp_len"] = [len(x.split()) for x in p]
res["ratio"] = res["hyp_len"] / res["ref_len"].clip(lower=1)
print("empty preds:", (res["hyp_len"] == 0).sum(),
      "| ratio>2:", (res["ratio"] > 2).sum(),
      "| ratio<0.5:", (res["ratio"] < 0.5).sum())

res["sample_wer"] = [jiwer.wer(a, b) if a else np.nan for a, b in zip(r, p)]
print(res["sample_wer"].describe())
print(res.sort_values("sample_wer", ascending=False)[["reference", "prediction", "sample_wer"]].head(10).to_string())

# loop collapse (same function as Whisper)
p_clean = [collapse_repeats(x) for x in p]
res["looped"] = [a != b for a, b in zip(p, p_clean)]
res["prediction_clean"] = p_clean
print("looped clips:", res["looped"].sum())
print("strict   :", score(res["reference"], res["prediction"]))
print("collapsed:", score(res["reference"], res["prediction_clean"]))

res["len_bin"] = pd.cut(res["ref_len"], [0, 2, 5, 10, 20, 200])
print(breakdown(res, "len_bin").to_string(index=False))
print(res.groupby("looped")["duration"].describe())

In [24]:
import json, platform, torch, transformers, importlib.metadata as im

overall = score(res["reference"], res["prediction"])
json.dump({
    "model": "Hindi-ASR-Qwen3-0.6B-Finetuned", "n_samples": len(res), **overall,
    "wer_loop_collapsed": score(res["reference"], res["prediction_clean"])["WER"],
    "decoding": "qwen-asr default, language=Hindi forced, fp16",
    "normalization": "NFC, strip ZWJ/ZWNJ, strip nukta, lowercase, drop non L/M/N chars",
    "versions": {"transformers": transformers.__version__, "torch": torch.__version__,
                 "jiwer": im.version("jiwer"), "python": platform.python_version()},
}, open("metrics_qwen.json", "w"), indent=2, ensure_ascii=False)
print(open("metrics_qwen.json").read())